# أثر — بناء فهرس التعرّف على المواقع

**هذا الدفتر لا يُدرّب نموذجًا.** ولا يحتاج معرفة سابقة بتعلّم الآلة.

---

## ما الذي يحدث هنا بالضبط

نستخدم نموذجًا جاهزًا اسمه **DINOv3** دربته Meta على 1.7 مليار صورة.
وظيفته الوحيدة عندنا: أن يأخذ صورة ويُخرج **قائمة من 384 رقمًا** تصف شكلها.

هذه القائمة تُسمّى *متجهًا* (embedding). والخاصية المفيدة فيها:

> صورتان لنفس الواجهة الصخرية تُعطيان قائمتَي أرقام **متقاربتين**،
> حتى لو اختلفت الزاوية والإضاءة والمسافة.

فيتحول سؤال «ما هذا الموقع؟» إلى عملية حسابية بسيطة:
احسب متجه صورة الكاميرا، وقارنه بمتجهات صورك المرجعية، وخذ الأقرب.

## لماذا لا نُدرّب؟

لأن التدريب هنا سيضرّ لا ينفع. لو دربنا مصنِّفًا على أربعة مواقع، فإن إضافة
موقع خامس تعني إعادة التدريب من الصفر. أما بهذه الطريقة فإضافة موقع =
إضافة صوره وإعادة تشغيل هذا الدفتر. دقيقتان.

## ماذا ستحصل عليه في النهاية

ملف واحد اسمه `reference-index.json` تضعه في مجلد `public/` في المشروع.
حجمه بضع مئات من الكيلوبايتات، ويحتوي الأرقام فقط لا الصور — فلا يُنزّل
زائر التطبيق صور المرجع إطلاقًا.

## هل أحتاج GPU؟

**لا.** بضع مئات من الصور تُعالَج على المعالج العادي في دقائق.
اترك إعداد Kaggle على `CPU` ولا تستهلك حصّتك من كرت الشاشة.

---
# ١ · تثبيت المكتبات

شغّل هذه الخلية أولًا. تأخذ نحو دقيقة.

> **كيف تُشغّل خلية؟** اضغط داخلها ثم `Shift + Enter`.

In [ ]:
!pip install -q transformers onnxruntime huggingface_hub pillow numpy
print('تم التثبيت ✓')

---
# ٢ · رفع صورك

في Kaggle، الصور تُرفع كـ **Dataset**:

1. من الشريط الأيمن: **Add Input** → **Upload** → **New Dataset**
2. ارفع مجلدًا بهذا الترتيب بالضبط:

```
athr-photos/
├── jubbah/          ← معرّف الموقع كما هو في sites.js حرفًا بحرف
│   ├── IMG_001.jpg
│   ├── IMG_002.jpg
│   └── ...           (١٥–٣٠ صورة)
├── qishlah/
├── aja/
├── museum/
└── _negatives/      ← مهم جدًا، اقرأ أدناه
```

### أسماء المجلدات

يجب أن تطابق `id` في `src/data/sites.js`:
`jubbah` · `qishlah` · `aja` · `museum`

### ما هو `_negatives` ولماذا هو مهم؟

صور **لا تخصّ أي موقع**: رمل، سماء، صخر عادي، سيارتك، حذاؤك، جدار فندق.
خمسون صورة تكفي.

بدونها يصبح النموذج واثقًا من كل شيء — فيقول عن صورة سماء إنها «جبة
بنسبة ٧٠٪». نستخدم هذه الصور لضبط العتبة التي يقول عندها التطبيق
**«لا أعرف»**، وهي أهم جملة في أي عرض أمام لجنة تحكيم.

### كيف تصوّر

| افعل | لا تفعل |
|---|---|
| زوايا مختلفة لكل واجهة | ٣٠ صورة من نفس الوقفة |
| صباحًا **و** ظهرًا (الضوء المائل يغيّر النقش كليًا) | إضاءة واحدة فقط |
| قريب ومتوسط وبعيد | مقاس واحد |
| بعض الصور غير مثالية: مائلة، جزئية، فيها اهتزاز | صور احترافية فقط |

السبب في العمود الأخير: السائح لن يصوّر باحتراف. درّب الفهرس على ما
سيحدث فعلًا، لا على ما تتمنّاه.

In [ ]:
from pathlib import Path

# ⬇️ عدّل هذا السطر ليطابق اسم الـ Dataset الذي رفعته
PHOTOS_DIR = Path('/kaggle/input/athr-photos')

# معرّفات المواقع كما في src/data/sites.js
SITE_IDS = ['jubbah', 'qishlah', 'aja', 'museum']
NEGATIVES_DIR = '_negatives'

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.JPG', '.JPEG', '.PNG'}

def list_images(folder):
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob('*') if p.suffix in IMAGE_SUFFIXES)

# جرد سريع قبل أي شيء آخر — أرخص من اكتشاف النقص بعد نصف ساعة
print('المسار:', PHOTOS_DIR, '\n')
total = 0
for site_id in SITE_IDS:
    images = list_images(PHOTOS_DIR / site_id)
    total += len(images)
    flag = '✓' if len(images) >= 15 else '⚠️  أقل من ١٥ صورة'
    print(f'{site_id:12s} {len(images):3d} صورة   {flag}')

negatives = list_images(PHOTOS_DIR / NEGATIVES_DIR)
print(f'{NEGATIVES_DIR:12s} {len(negatives):3d} صورة   ' + ('✓' if len(negatives) >= 20 else '⚠️  أضف المزيد'))
print(f'\nالمجموع: {total + len(negatives)} صورة')

---
# ٣ · تحميل النموذج

**نقطة حاسمة:** نُنزّل هنا **نفس ملف النموذج بالضبط** الذي يُنزّله المتصفح.

لو بنينا الفهرس بنموذج وقارنّا في المتصفح بنموذج آخر، فالأرقام لا تعني
الشيء نفسه والنتائج ستكون عشوائية تمامًا. لذلك القيمتان أدناه يجب أن
تطابقا ما في `src/lib/recognition.local.js`.

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

# ⚠️ يجب أن تطابق MODEL_ID و MODEL_DTYPE في src/lib/recognition.local.js
MODEL_ID = 'onnx-community/dinov3-vits16-pretrain-lvd1689m-ONNX'
DTYPE = 'q8'

# transformers.js يسمّي ملفات ONNX حسب الدقّة بهذا الشكل
DTYPE_TO_FILE = {
    'fp32': 'onnx/model.onnx',
    'fp16': 'onnx/model_fp16.onnx',
    'q8':   'onnx/model_quantized.onnx',
    'int8': 'onnx/model_int8.onnx',
}

available = [f for f in list_repo_files(MODEL_ID) if f.endswith('.onnx')]
print('ملفات ONNX المتاحة في هذا النموذج:')
for f in available:
    print('   ', f)

wanted = DTYPE_TO_FILE[DTYPE]
if wanted not in available:
    raise SystemExit(
        f'\n❌ الملف {wanted} غير موجود.\n'
        f'اختر دقّة من القائمة أعلاه، وغيّر DTYPE هنا و MODEL_DTYPE في التطبيق معًا.'
    )

onnx_path = hf_hub_download(MODEL_ID, wanted)
print(f'\n✓ نزّلنا: {wanted}')

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image
from transformers import AutoImageProcessor

# المعالِج يقصّ الصورة ويغيّر مقاسها ويعدّل ألوانها تمامًا كما يفعل
# المتصفح. استخدام معالِج مختلف يفسد المطابقة بصمت.
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

INPUT_NAME = session.get_inputs()[0].name
OUTPUT_NAMES = [o.name for o in session.get_outputs()]
print('مدخل النموذج :', INPUT_NAME)
print('مخرجاته      :', OUTPUT_NAMES)

---
# ٤ · دالة تحويل الصورة إلى متجه

خطوتان:

1. **الاستخلاص** — نمرّر الصورة على النموذج ونأخذ رمز `CLS`، وهو الرقم
   الذي يلخّص الصورة كلها (بقية الأرقام تصف رقعًا صغيرة منها).
2. **التطبيع** — نجعل طول المتجه = ١. بعدها يصير حساب التشابه مجرد ضرب
   نقطي، والنتيجة بين ١- و١+ حيث ١ = متطابق تمامًا.

هذا يطابق سطرًا بسطر ما تفعله دالة `toVector` في التطبيق.

In [ ]:
def embed(image_path):
    """يحوّل صورة إلى متجه مطبَّع (384 رقمًا لنموذج ViT-S)."""
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors='np')
    outputs = session.run(None, {INPUT_NAME: inputs['pixel_values'].astype(np.float32)})
    result = dict(zip(OUTPUT_NAMES, outputs))

    if 'pooler_output' in result:
        vector = result['pooler_output'][0]
    else:
        # [batch, tokens, hidden] → الرمز الأول (CLS) يلخّص الصورة
        vector = result[OUTPUT_NAMES[0]][0][0]

    vector = np.asarray(vector, dtype=np.float32).ravel()
    return vector / (np.linalg.norm(vector) + 1e-10)


# اختبار سريع على أول صورة متاحة
_first = next((p for s in SITE_IDS for p in list_images(PHOTOS_DIR / s)), None)
if _first is None:
    raise SystemExit('❌ لم نجد أي صورة. راجع الخطوة ٢.')

_v = embed(_first)
print(f'✓ نجحت التجربة على: {_first.name}')
print(f'  طول المتجه : {len(_v)} رقمًا')
print(f'  أول ٥ أرقام: {np.round(_v[:5], 4)}')

---
# ٥ · حساب متجهات كل الصور

قد يستغرق هذا بضع دقائق حسب عدد الصور. اتركه يعمل.

In [ ]:
import time

entries = []      # المتجهات المرجعية (المواقع)
neg_vectors = []  # متجهات الصور السلبية (لضبط العتبة)

start = time.time()

for site_id in SITE_IDS:
    images = list_images(PHOTOS_DIR / site_id)
    for path in images:
        try:
            entries.append({'siteId': site_id, 'photo': path.name, 'vector': embed(path)})
        except Exception as error:
            print(f'⚠️  تخطّينا {path.name}: {error}')
    print(f'{site_id:12s} ✓ {len(images)} صورة')

for path in list_images(PHOTOS_DIR / NEGATIVES_DIR):
    try:
        neg_vectors.append(embed(path))
    except Exception:
        pass

print(f'\nالمجموع: {len(entries)} متجهًا مرجعيًا + {len(neg_vectors)} سلبيًا')
print(f'الزمن  : {time.time() - start:.0f} ثانية')

---
# ٦ · هل يعمل فعلًا؟ (أهم خلية في الدفتر)

لا تُصدّق أن النظام يعمل لأنه لم يُعطِ خطأً. **قِسه.**

الطريقة تُسمّى *اترك-واحدًا-خارجًا*: نأخذ كل صورة على حدة، نُخفيها من
الفهرس، ثم نسأل: بأي موقع سيُطابقها النظام وهي غائبة عن مرجعه؟

هذا يحاكي ما سيحدث فعلًا: صورة جديدة لم يرها النظام من قبل.

**اقرأ النتيجة هكذا:**

| الدقّة | الحكم |
|---|---|
| ٩٥٪ فأكثر | ممتاز، اعرضه بثقة |
| ٨٥–٩٥٪ | جيد، وسيتحسن بصور أكثر تنوّعًا |
| أقل من ٨٥٪ | راجع الجدول أسفل الخلية لتعرف أي موقع يُربكه |

In [ ]:
from collections import defaultdict

matrix = np.stack([e['vector'] for e in entries])
labels = [e['siteId'] for e in entries]

correct = 0
positive_scores = []   # درجات المطابقات الصحيحة — نحتاجها لضبط العتبة
confusion = defaultdict(lambda: defaultdict(int))

for i in range(len(entries)):
    scores = matrix @ matrix[i]
    scores[i] = -np.inf  # نُخفي الصورة نفسها

    # أعلى درجة لكل موقع، لا لكل صورة
    best_per_site = defaultdict(lambda: -np.inf)
    for j, score in enumerate(scores):
        if j != i and score > best_per_site[labels[j]]:
            best_per_site[labels[j]] = score

    ranked = sorted(best_per_site.items(), key=lambda kv: -kv[1])
    predicted, top_score = ranked[0]

    confusion[labels[i]][predicted] += 1
    if predicted == labels[i]:
        correct += 1
        positive_scores.append(top_score)

accuracy = correct / len(entries)
print(f'الدقّة: {accuracy:.1%}  ({correct} من {len(entries)})\n')

print('أين يُخطئ — الصفوف = الحقيقة، الأعمدة = التوقّع')
header = 'الحقيقة \\ التوقّع'.ljust(18) + ''.join(s.ljust(11) for s in SITE_IDS)
print(header)
for actual in SITE_IDS:
    row = actual.ljust(18)
    for predicted in SITE_IDS:
        count = confusion[actual][predicted]
        mark = f'{count}' if actual == predicted else (f'{count} ←' if count else '·')
        row += mark.ljust(11)
    print(row)

print('\nالأرقام خارج القطر هي الأخطاء. لو تركّزت في موقع واحد، صوّره أكثر.')

---
# ٧ · ضبط عتبة «لا أعرف»

الآن نستخدم الصور السلبية. الفكرة:

- الصور **الصحيحة** يجب أن تتجاوز العتبة
- الصور **السلبية** (رمل، سماء) يجب أن تسقط تحتها

نختار العتبة بحيث لا تتجاوزها أي صورة سلبية تقريبًا — لأن **الخطأ الواثق
أسوأ من الاعتراف بالجهل**، خصوصًا أمام لجنة تحكيم.

In [ ]:
positive_scores = np.array(positive_scores)

if len(neg_vectors) == 0:
    print('⚠️  لا توجد صور سلبية — لا يمكن ضبط العتبة بثقة.')
    print('    ارجع للخطوة ٢ وأضف مجلد _negatives. الأمر يستحق نصف ساعة.')
else:
    negative_scores = np.array([ (matrix @ v).max() for v in neg_vectors ])

    print(f'الصور الصحيحة  ({len(positive_scores):3d}): '
          f'أدنى {positive_scores.min():.3f} · وسيط {np.median(positive_scores):.3f}')
    print(f'الصور السلبية  ({len(negative_scores):3d}): '
          f'أعلى {negative_scores.max():.3f} · وسيط {np.median(negative_scores):.3f}')

    # العتبة المقترحة: أعلى بقليل من أقوى صورة سلبية
    suggested = float(np.percentile(negative_scores, 99)) + 0.02
    kept = (positive_scores >= suggested).mean()

    print(f'\n➜ العتبة المقترحة: {suggested:.2f}')
    print(f'   ستقبل {kept:.0%} من الصور الصحيحة وترفض ~99% من السلبية.')

    if kept < 0.8:
        print('\n⚠️  العتبة تقصي كثيرًا من الصور الصحيحة.')
        print('    السبب غالبًا تشابه الصور السلبية مع مواقعك (صخور مشابهة).')
        print('    الحل: صور مرجعية أكثر تنوّعًا، لا خفض العتبة.')

    print(f'\nضع هذا الرقم في SIMILARITY_THRESHOLD داخل src/lib/recognition.local.js')

---
# ٨ · تصدير الفهرس

نكتب `reference-index.json`. نُدوّر الأرقام إلى ٥ منازل عشرية لتقليل
الحجم دون أثر ملموس على الدقّة.

In [ ]:
import json, datetime

payload = {
    # هذه البصمة يتحقّق منها التطبيق ويرفض الفهرس إن لم تطابق نموذجه
    'model': MODEL_ID,
    'dtype': DTYPE,
    'dims': int(matrix.shape[1]),
    'createdAt': datetime.datetime.utcnow().strftime('%Y-%m-%d'),
    'accuracy': round(float(accuracy), 4),
    'entries': [
        {
            'siteId': e['siteId'],
            'photo': e['photo'],
            'vector': [round(float(x), 5) for x in e['vector']],
        }
        for e in entries
    ],
}

out_path = '/kaggle/working/reference-index.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, separators=(',', ':'))

size_kb = len(json.dumps(payload)) / 1024
print(f'✓ كُتب الملف: {out_path}')
print(f'  {len(entries)} متجهًا · {size_kb:.0f} كيلوبايت')
print('\nنزّله من لوحة Output على يمين الشاشة ⬇️')

---
# ٩ · ماذا تفعل بالملف

1. نزّل `reference-index.json` من لوحة **Output** يمين الصفحة
2. ضعه في مجلد `public/` في المشروع
3. في `src/lib/recognition.local.js` عدّل `SIMILARITY_THRESHOLD` إلى الرقم
   الذي اقترحته الخطوة ٧
4. في ملف `.env`:
   ```
   VITE_RECOGNITION_PROVIDER=local
   ```
5. ```bash
   npm install @huggingface/transformers
   npm run dev
   ```

التعرّف الآن حقيقي، ويعمل داخل المتصفح، وبلا إنترنت بعد التحميل الأول.

---

## متى تعيد تشغيل هذا الدفتر

- أضفت صورًا جديدة أو موقعًا جديدًا
- غيّرت `MODEL_ID` أو `DTYPE` في التطبيق ← **إلزامي**، وإلا فسدت المطابقة

## إن كانت الدقّة منخفضة

الترتيب الصحيح للعلاج:

1. **صور أكثر تنوّعًا** — أكبر أثر بفارق كبير، وأرخص خطوة
2. راجع جدول الأخطاء في الخطوة ٦: أي موقعين يختلطان؟ صوّرهما أكثر
3. جرّب `DTYPE = 'fp32'` (أدق، لكن التنزيل أثقل على الجوال)
4. وأخيرًا فقط: نموذج أكبر `dinov3-vitl16` — بطيء على الهاتف، لا تبدأ به

**لا تبدأ من النموذج.** المشكلة في مشروعك ستكون في الصور بنسبة ٩٠٪.